In [14]:
import requests
from bs4 import BeautifulSoup

url = "http://www.bspider.top/anjuke/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
}

r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding

soup = BeautifulSoup(r.text, "html.parser")

print("status :", r.status_code)
print("title  :", soup.title.get_text(strip=True))


status : 200
title  : 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客


In [15]:
rows = soup.select("div.property")
print("房源条数:", len(rows))
print("第一条的 class:", rows[0].get("class"))


房源条数: 60
第一条的 class: ['property']


In [16]:
row = rows[0]

def text(sel):
    node = row.select_one(sel)
    return node.get_text(strip=True) if node else ""

info = row.select("div.property-content-info")
items = info[0].select("p.property-content-info-text")

print("title   :", text("h3"))
print("house   :", text(".property-content-info-comm-name"))
print("address :", text(".property-content-info-comm-address"))
print("struct  :", items[0].get_text(strip=True))
print("area    :", items[1].get_text(strip=True))
print("floor   :", items[3].get_text(strip=True))
print("maketime:", items[4].get_text(strip=True))
print("price   :", text(".property-price-average"))


title   : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house   : 万科金域学府翰林菁华里
address : 南岸学府大道二塘路59号
struct  : 3室2厅1卫
area    : 89.9㎡
floor   : 低层(共27层)
maketime: 2019年建造
price   : 16465元/㎡


In [4]:
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

def parse_property(row):
    def text(sel):
        node = row.select_one(sel)
        return node.get_text(strip=True) if node else ""

    info = row.select("div.property-content-info")
    items = info[0].select("p.property-content-info-text") if info else []

    return {
        "title":    text("h3"),
        "house":    text(".property-content-info-comm-name"),
        "address":  text(".property-content-info-comm-address"),
        "struct":   items[0].get_text(strip=True) if len(items) > 0 else "",
        "area":     items[1].get_text(strip=True) if len(items) > 1 else "",
        "floor":    items[3].get_text(strip=True) if len(items) > 3 else "",
        "maketime": items[4].get_text(strip=True) if len(items) > 4 else "",
        "price":    text(".property-price-average"),
    }

for idx in [0, 1, len(rows) - 1]:
    print(f"--- 第 {idx+1} 条 ---")
    for k in FIELDS:
        print(f"  {k:9}:", parse_property(rows[idx])[k])


--- 第 1 条 ---
  title    : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house    : 万科金域学府翰林菁华里
  address  : 南岸学府大道二塘路59号
  struct   : 3室2厅1卫
  area     : 89.9㎡
  floor    : 低层(共27层)
  maketime : 2019年建造
  price    : 16465元/㎡
--- 第 2 条 ---
  title    : 黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住
  house    : 两代一家
  address  : 渝北渝北黄泥磅紫荆路29号
  struct   : 3室2厅2卫
  area     : 94.5㎡
  floor    : 低层(共28层)
  maketime : 2008年建造
  price    : 14815元/㎡
--- 第 60 条 ---
  title    : 精装修 电梯房 4室2厅 91平
  house    : 华润凯旋天地
  address  : 南岸弹子石腾龙大道45号
  struct   : 4室2厅2卫
  area     : 91.3㎡
  floor    : 低层(共32层)
  maketime : 2015年建造
  price    : 15886元/㎡


In [17]:
import time
from urllib.parse import urljoin, urlparse

def crawl(max_pages):
    results, visited = [], set()
    current_url, page_no = url, 1

    while True:
        if page_no > max_pages:
            print(f"已达本次页数上限 {max_pages}，停止")
            break

        r = requests.get(current_url, headers=headers, timeout=30)
        r.raise_for_status()
        r.encoding = r.apparent_encoding
        sp = BeautifulSoup(r.text, "html.parser")

        page_rows = sp.select("div.property")
        if not page_rows:
            print("本页 0 条 → 结构异常，中止"); break

        results.extend(parse_property(row) for row in page_rows)
        print(f"第 {page_no:2d} 页 | 本页 {len(page_rows):2d} 条 | 累计 {len(results):3d} | {current_url}")
        visited.add(current_url)

        pagination = sp.select_one("div.pagination")
        nxt = sp.select_one("a.next.next-active")

        if pagination is None:
            print("分页区域消失 → 结构异常，中止"); break
        if nxt is None or nxt.get("href") in (None, "#"):
            print("「下一页」已禁用 → 到达末页，正常结束"); break

        next_url = urljoin(current_url, nxt["href"])
        if next_url in visited:
            print("下一页地址重复 → 中止"); break
        if urlparse(next_url).netloc != "www.bspider.top":
            print("地址越出教学站 → 中止"); break

        current_url, page_no = next_url, page_no + 1
        time.sleep(1)

    return results

test = crawl(2)

pairs = [(h["title"], h["house"]) for h in test]
print("试采:", len(test), "条 | 第1页与第2页重复:", len(set(pairs[:60]) & set(pairs[60:])))


第  1 页 | 本页 60 条 | 累计  60 | http://www.bspider.top/anjuke/
第  2 页 | 本页 60 条 | 累计 120 | http://www.bspider.top/anjuke/?page=2
已达本次页数上限 2，停止
试采: 120 条 | 第1页与第2页重复: 0


In [18]:
all_houses = crawl(20)
print("最终:", len(all_houses), "条")


第  1 页 | 本页 60 条 | 累计  60 | http://www.bspider.top/anjuke/
第  2 页 | 本页 60 条 | 累计 120 | http://www.bspider.top/anjuke/?page=2
第  3 页 | 本页 60 条 | 累计 180 | http://www.bspider.top/anjuke/?page=3
第  4 页 | 本页 60 条 | 累计 240 | http://www.bspider.top/anjuke/?page=4
第  5 页 | 本页 60 条 | 累计 300 | http://www.bspider.top/anjuke/?page=5
第  6 页 | 本页 60 条 | 累计 360 | http://www.bspider.top/anjuke/?page=6
第  7 页 | 本页 60 条 | 累计 420 | http://www.bspider.top/anjuke/?page=7
第  8 页 | 本页 60 条 | 累计 480 | http://www.bspider.top/anjuke/?page=8
第  9 页 | 本页 60 条 | 累计 540 | http://www.bspider.top/anjuke/?page=9
第 10 页 | 本页 60 条 | 累计 600 | http://www.bspider.top/anjuke/?page=10
「下一页」已禁用 → 到达末页，正常结束
最终: 600 条


In [19]:
import csv
import os

seen, unique_houses = set(), []
for h in all_houses:
    key = tuple(h[k] for k in FIELDS)
    if key not in seen:
        seen.add(key)
        unique_houses.append(h)

print("去重前:", len(all_houses), "| 去重后:", len(unique_houses))

with open("data/houseinfo.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    for h in unique_houses:
        writer.writerow([h[k] for k in FIELDS])

print("已保存 data/houseinfo.csv,", os.path.getsize("data/houseinfo.csv"), "字节")


去重前: 600 | 去重后: 498
已保存 data/houseinfo.csv, 94453 字节


In [20]:
with open("data/houseinfo.csv", encoding="utf-8-sig", newline="") as f:
    data_back = list(csv.DictReader(f))

print("回读数据行数:", len(data_back), "（表头不计）")
print("表头:", list(data_back[0].keys()))
for i in [0, 300, len(data_back) - 1]:
    print(f"--- 第 {i+1} 行 ---")
    print(data_back[i])


回读数据行数: 498 （表头不计）
表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
--- 第 1 行 ---
{'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}
--- 第 301 行 ---
{'title': '杨家坪 超人气楼盘地段 总价150万左右买九龙半岛品质洋房，', 'house': '荣安九龙壹号', 'address': '九龙坡杨家坪龙江支路,近龙江路', 'struct': '3室2厅2卫', 'area': '90㎡', 'floor': '中层(共8层)', 'maketime': '2021年建造', 'price': '16889元/㎡'}
--- 第 498 行 ---
{'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}


In [21]:
import csv
import pymysql

FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

with open("data/houseinfo.csv", encoding="utf-8-sig", newline="") as f:
    records = list(csv.DictReader(f))

print("CSV 行数:", len(records))
print("表头:", list(records[0].keys()))
print("表头顺序与 FIELDS 一致:", list(records[0].keys()) == FIELDS)

conn = pymysql.connect(
    host="mysql",
    port=3306,
    user="student",
    password="123456",
    database="smartpark_lab",
    charset="utf8mb4",
    autocommit=False,
)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM houseinfo")
print("连接成功，入库前条数:", cur.fetchone()[0])

cur.close()
conn.close()


CSV 行数: 498
表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
表头顺序与 FIELDS 一致: True
连接成功，入库前条数: 498


In [22]:
conn = pymysql.connect(
    host="mysql", port=3306, user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4", autocommit=False,
)
cur = conn.cursor()

sql = """
INSERT INTO houseinfo (title, house, address, struct, area, floor, maketime, price)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

sample = [tuple(r[k] for k in FIELDS) for r in records[:5]]

try:
    n = cur.executemany(sql, sample)
    conn.commit()
    print("本次写入:", n, "条")
    cur.execute("SELECT COUNT(*) FROM houseinfo")
    print("当前表内条数:", cur.fetchone()[0])
except Exception as e:
    conn.rollback()
    print("出错，已回滚:", e)

cur.close()
conn.close()


本次写入: 5 条
当前表内条数: 503


In [25]:
conn = pymysql.connect(
    host="mysql", port=3306, user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4", autocommit=False,
)
cur = conn.cursor()

cur.execute(
    "SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo"
)
existing = set(cur.fetchall())

sql = """
INSERT INTO houseinfo (title, house, address, struct, area, floor, maketime, price)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

all_rows = [tuple(r[k] for k in FIELDS) for r in records]
new_rows = [t for t in all_rows if t not in existing]
print(f"CSV 共 {len(all_rows)} 条 | 库内已有 {len(existing)} 条 | 跳过重复 {len(all_rows)-len(new_rows)} 条 | 待插入 {len(new_rows)} 条")

try:
    if new_rows:
        n = cur.executemany(sql, new_rows)
        conn.commit()
        print("本次新增:", n, "条")
    else:
        print("没有新记录，不执行插入")
    cur.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库后总条数:", cur.fetchone()[0])
except Exception as e:
    conn.rollback()
    print("出错，已回滚:", e)

cur.close()
conn.close()


CSV 共 498 条 | 库内已有 498 条 | 跳过重复 498 条 | 待插入 0 条
没有新记录，不执行插入
入库后总条数: 498


In [24]:
conn = pymysql.connect(
    host="mysql", port=3306, user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4", autocommit=False,
)
cur = conn.cursor()

cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo ORDER BY id")
db_rows = cur.fetchall()

samples = [0, 479, 497]

for i in samples:
    csv_row = tuple(records[i][k] for k in FIELDS)
    db_row = db_rows[i]
    print(f"--- 样本: CSV 第 {i+1} 行 / 数据库 id={i+1} ---")
    ok = True
    for f, c, d in zip(FIELDS, csv_row, db_row):
        same = (c == d)
        ok = ok and same
        print(f"  {f:8s} CSV[{c}] DB[{d}] {'OK' if same else '<-- 不一致'}")
    print("  ==> 该样本全部一致" if ok else "  ==> 有字段不一致！")

cur.close()
conn.close()


--- 样本: CSV 第 1 行 / 数据库 id=1 ---
  title    CSV[万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器] DB[万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器] OK
  house    CSV[万科金域学府翰林菁华里] DB[万科金域学府翰林菁华里] OK
  address  CSV[南岸学府大道二塘路59号] DB[南岸学府大道二塘路59号] OK
  struct   CSV[3室2厅1卫] DB[3室2厅1卫] OK
  area     CSV[89.9㎡] DB[89.9㎡] OK
  floor    CSV[低层(共27层)] DB[低层(共27层)] OK
  maketime CSV[2019年建造] DB[2019年建造] OK
  price    CSV[16465元/㎡] DB[16465元/㎡] OK
  ==> 该样本全部一致
--- 样本: CSV 第 480 行 / 数据库 id=480 ---
  title    CSV[恒大山水城 精装修 三房一卫 住家安静舒适] DB[恒大山水城 精装修 三房一卫 住家安静舒适] OK
  house    CSV[恒大照母山山水城] DB[恒大照母山山水城] OK
  address  CSV[渝北照母山恒康路9号] DB[渝北照母山恒康路9号] OK
  struct   CSV[3室2厅1卫] DB[3室2厅1卫] OK
  area     CSV[90.2㎡] DB[90.2㎡] OK
  floor    CSV[低层(共32层)] DB[低层(共32层)] OK
  maketime CSV[2014年建造] DB[2014年建造] OK
  price    CSV[15530元/㎡] DB[15530元/㎡] OK
  ==> 该样本全部一致
--- 样本: CSV 第 498 行 / 数据库 id=498 ---
  title    CSV[恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售] DB[恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售] OK
  house    CSV[恒大照母山山水城] DB[恒大照母山山水城] OK
  address  CSV[渝北照母